# 03 - DPO Training

**Project**: LLM Fine-Tuning with DPO  
**Author**: Pranay Hedau  
**Runs on**: Kaggle T4 GPU (15GB VRAM)  

### What this notebook does
Runs Direct Preference Optimization (DPO) on top of our SFT model from notebook 02.
The SFT model plays two roles simultaneously:
- **Reference model** (frozen) — the anchor, tells DPO how far the policy has drifted
- **Policy model** (trainable) — the model we are actually improving

After training, the policy model generates responses that are more aligned
with human preferences than the SFT baseline.

### Kaggle Setup Checklist
Before running:
- [ ] GPU T4 x1 enabled
- [ ] Internet enabled
- [ ] HF_TOKEN added as Kaggle Secret
- [ ] `ultrafeedback_10k.jsonl` dataset attached
- [ ] `sft_adapter/` dataset attached (output from notebook 02)

## 1. Environment Check

In [ ]:
import os
import torch

print('=== ENVIRONMENT CHECK ===')
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'Total VRAM      : {total_vram:.1f} GB')
else:
    raise RuntimeError('No GPU detected. Enable T4 via Settings → Accelerator.')

# Load HF token from Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN        : Found via Kaggle Secrets')
except Exception:
    from dotenv import load_dotenv
    load_dotenv('../.env')
    HF_TOKEN = os.getenv('HF_TOKEN')
    print('HF_TOKEN        : Loaded from .env')

if not HF_TOKEN:
    raise ValueError('HF_TOKEN not found.')

print('All checks passed.')

## 2. Install Dependencies

In [ ]:
import subprocess
packages = [
    'trl>=0.8.6',
    'peft>=0.10.0',
    'bitsandbytes>=0.43.0',
    'accelerate>=0.29.0',
    'transformers>=4.40.0',
    'datasets>=2.19.0',
]
subprocess.run(['pip', 'install', '-q'] + packages, check=True)
print('Dependencies installed.')

## 3. Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, PeftModel, TaskType
from trl import DPOTrainer, DPOConfig
from huggingface_hub import login

print('Imports complete.')

## 4. Configuration

These are the knobs we tune in our ablation study (notebook 04).
For the first full run we are using well-established defaults from the DPO paper.

In [ ]:
# ── Model paths ───────────────────────────────────────────────
BASE_MODEL_ID   = 'meta-llama/Llama-3.2-3B-Instruct'
SFT_ADAPTER_DIR = '/kaggle/input/sft-adapter/sft_adapter'
DPO_OUTPUT_DIR  = '/kaggle/working/dpo_model'

# ── DPO hyperparameters ───────────────────────────────────────
# beta: controls how far the policy can drift from the reference.
# 0.1 is the value used in the original DPO paper (Rafailov et al. 2023).
# Lower beta = more aggressive alignment but risks forgetting SFT knowledge.
# Higher beta = conservative, stays close to reference, weaker alignment.
BETA            = 0.1

# ── QLoRA ─────────────────────────────────────────────────────
# Same rank as SFT for consistency in our ablation comparisons
LORA_RANK       = 64
LORA_ALPHA      = 16
LORA_DROPOUT    = 0.05

# ── Training ──────────────────────────────────────────────────
MAX_SEQ_LEN     = 1024
MAX_PROMPT_LEN  = 512    # prompt gets half the budget, response gets the other half
NUM_EPOCHS      = 1
BATCH_SIZE      = 2      # DPO needs 2x memory vs SFT (processes chosen + rejected)
GRAD_ACCUM      = 8      # effective batch = 2 * 8 = 16
LR              = 5e-5   # lower than SFT — DPO needs finer updates
WARMUP_RATIO    = 0.1    # longer warmup for DPO stability
SAVE_STEPS      = 100
LOGGING_STEPS   = 25

print('DPO Configuration:')
print(f'  Base model      : {BASE_MODEL_ID}')
print(f'  SFT adapter     : {SFT_ADAPTER_DIR}')
print(f'  Beta            : {BETA}  (DPO divergence penalty)')
print(f'  LoRA rank       : {LORA_RANK}')
print(f'  Learning rate   : {LR}  (lower than SFT for stability)')
print(f'  Effective batch : {BATCH_SIZE * GRAD_ACCUM}')
print()
print('NOTE: DPO uses batch_size=2 vs SFT batch_size=4 because')
print('DPOTrainer processes both chosen AND rejected for each example,')
print('so memory usage is roughly 2x that of SFT.')

## 5. Load & Format Dataset for DPO

DPOTrainer expects exactly 3 fields: `prompt`, `chosen`, `rejected`.
The chosen and rejected fields must be **just the assistant response text**,
not the full conversation. We extract that here.

In [ ]:
def extract_assistant_text(field):
    """Extract the assistant response text from chosen/rejected field."""
    if isinstance(field, list):
        for turn in field:
            if turn.get('role') == 'assistant':
                return turn.get('content', '')
        return field[-1].get('content', '') if field else ''
    return str(field)

# Load our 10K subset
DATA_PATH = '/kaggle/input/ultrafeedback-10k/ultrafeedback_10k.jsonl'
if not os.path.exists(DATA_PATH):
    DATA_PATH = '../data/ultrafeedback_10k.jsonl'

raw_dataset = load_dataset('json', data_files=DATA_PATH, split='train')

def format_dpo(example):
    """
    DPOTrainer expects:
      prompt   : the instruction (plain string)
      chosen   : the better response (plain string, assistant text only)
      rejected : the worse response (plain string, assistant text only)

    Why plain strings and not chat-formatted?
    DPOTrainer applies the chat template internally using the tokenizer.
    Passing pre-formatted text causes double-formatting bugs.
    """
    return {
        'prompt'  : example['prompt'],
        'chosen'  : extract_assistant_text(example['chosen']),
        'rejected': extract_assistant_text(example['rejected']),
    }

dpo_dataset = raw_dataset.map(
    format_dpo,
    remove_columns=raw_dataset.column_names
)

# 90/10 split 90% training and 10% validation dataset
split      = dpo_dataset.train_test_split(test_size=0.1, seed=42)
train_data = split['train']
val_data   = split['test']

print(f'Train examples : {len(train_data)}')
print(f'Val examples   : {len(val_data)}')
print()
print('Sample entry:')
sample = train_data[0]
print(f'  prompt   : {sample["prompt"][:100]}...')
print(f'  chosen   : {sample["chosen"][:100]}...')
print(f'  rejected : {sample["rejected"][:100]}...')

## 6. Load Model

We load the base Llama 3.2 3B in 4-bit, then attach the SFT adapter on top.
This gives us the SFT model as our starting point for DPO.

**Why load the SFT adapter instead of training from scratch?**
DPO on top of a raw base model produces weaker alignment because the base
model has no instruction-following foundation. SFT provides that foundation
and DPO refines the preference on top of it.

In [ ]:
login(token=HF_TOKEN)

# 4-bit quantization config (same as notebook 02)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load tokenizer
print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'   # DPO requires left padding (unlike SFT)

# Load base model in 4-bit
print('Loading base model in 4-bit... (~2-3 minutes)')
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    token=HF_TOKEN,
    torch_dtype=torch.bfloat16,
)
base_model.config.use_cache = False

# Attach SFT adapter — this is now our policy model starting point
print('Attaching SFT adapter...')
model = PeftModel.from_pretrained(
    base_model,
    SFT_ADAPTER_DIR,
    is_trainable=True,    # policy model — weights WILL be updated
)

if torch.cuda.is_available():
    used = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU memory: {used:.1f} GB used / {total:.1f} GB total')

print('Policy model ready.')

In [ ]:
# Load reference model — same SFT adapter but FROZEN
# DPO needs this to compute the KL divergence penalty during training
# It answers: "how would the original SFT model have responded?"
print('Loading reference model (frozen SFT)...')
ref_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    token=HF_TOKEN,
    torch_dtype=torch.bfloat16,
)
ref_model = PeftModel.from_pretrained(
    ref_base,
    SFT_ADAPTER_DIR,
    is_trainable=False,   # reference model — weights NEVER change
)

if torch.cuda.is_available():
    used = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU memory after ref model: {used:.1f} GB / {total:.1f} GB')

print('Reference model ready (frozen).')
print()
print('Both models loaded:')
print('  Policy model    → will be updated by DPO')
print('  Reference model → frozen, acts as anchor')

## 7. DPO Training

**What to watch during training:**

| Metric | What it means | Healthy sign |
|--------|--------------|-------------|
| `train/loss` | Overall DPO loss | Decreasing |
| `train/rewards/chosen` | Model's score for chosen responses | Increasing |
| `train/rewards/rejected` | Model's score for rejected responses | Decreasing |
| `train/rewards/margins` | Gap between chosen and rejected scores | Increasing |
| `eval/loss` | Validation loss | Not increasing (no overfitting) |

The **rewards/margins** metric is the most important — it directly measures
whether DPO is separating good responses from bad ones.

In [ ]:
dpo_config = DPOConfig(
    output_dir=DPO_OUTPUT_DIR,
    beta=BETA,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type='cosine',
    bf16=True,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    eval_strategy='steps',
    eval_steps=SAVE_STEPS,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to='none',
    max_length=MAX_SEQ_LEN,
    max_prompt_length=MAX_PROMPT_LEN,
    # loss_type: 'sigmoid' is the standard DPO loss from the paper
    # alternatives: 'hinge', 'ipo' — we ablate this in notebook 04
    loss_type='sigmoid',
    remove_unused_columns=False,
)

trainer = DPOTrainer(
    model=model,
    ref_model=ref_model,
    args=dpo_config,
    train_dataset=train_data,
    eval_dataset=val_data,
    tokenizer=tokenizer,
)

print('DPOTrainer configured.')
print()
print('Key metrics to watch during training:')
print('  rewards/margins → should increase (model learning preference gap)')
print('  rewards/chosen  → should increase')
print('  rewards/rejected → should decrease')
print('  eval/loss       → should not increase (no overfitting)')

In [ ]:
# ── Run DPO Training ──────────────────────────────────────────
# Expected: ~2-3 hours on Kaggle T4
# (longer than SFT because DPO processes chosen + rejected per step)
print('Starting DPO training...')
print('Expected time: ~2-3 hours on T4')
print()

train_result = trainer.train()

print()
print('=== DPO TRAINING COMPLETE ===')
print(f'Final train loss : {train_result.training_loss:.4f}')
print(f'Total steps      : {train_result.global_step}')
print(f'Runtime          : {train_result.metrics["train_runtime"]:.0f}s  '
      f'({train_result.metrics["train_runtime"]/3600:.1f} hours)')

## 8. Log Training Metrics

We save the training history so we can plot it in notebook 04
for the ablation study and results analysis.

In [ ]:
import json
import matplotlib.pyplot as plt

# Extract training history from trainer logs
log_history = trainer.state.log_history

# Separate train and eval logs
train_logs = [l for l in log_history if 'loss' in l and 'eval_loss' not in l]
eval_logs  = [l for l in log_history if 'eval_loss' in l]

# Save to disk for notebook 04
os.makedirs(DPO_OUTPUT_DIR, exist_ok=True)
with open(f'{DPO_OUTPUT_DIR}/training_logs.json', 'w') as f:
    json.dump(log_history, f, indent=2)
print(f'Training logs saved to {DPO_OUTPUT_DIR}/training_logs.json')

# Quick training curves plot
if train_logs and eval_logs:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    # Loss curve
    train_steps  = [l['step'] for l in train_logs if 'loss' in l]
    train_losses = [l['loss'] for l in train_logs if 'loss' in l]
    eval_steps   = [l['step'] for l in eval_logs]
    eval_losses  = [l['eval_loss'] for l in eval_logs]

    axes[0].plot(train_steps, train_losses, color='#3498db', label='Train loss', linewidth=1.5)
    axes[0].plot(eval_steps,  eval_losses,  color='#e74c3c', label='Eval loss',  linewidth=1.5)
    axes[0].set_title('DPO Loss', fontweight='bold')
    axes[0].set_xlabel('Step')
    axes[0].set_ylabel('Loss')
    axes[0].legend()

    # Rewards margins
    margin_logs = [l for l in train_logs if 'rewards/margins' in l]
    if margin_logs:
        steps   = [l['step'] for l in margin_logs]
        margins = [l['rewards/margins'] for l in margin_logs]
        axes[1].plot(steps, margins, color='#2ecc71', linewidth=1.5)
        axes[1].set_title('Reward Margin\n(chosen - rejected)', fontweight='bold')
        axes[1].set_xlabel('Step')
        axes[1].set_ylabel('Margin')
        axes[1].axhline(0, color='gray', linestyle='--', linewidth=0.8)

    # Chosen vs rejected rewards
    chosen_logs   = [l for l in train_logs if 'rewards/chosen' in l]
    rejected_logs = [l for l in train_logs if 'rewards/rejected' in l]
    if chosen_logs:
        steps    = [l['step'] for l in chosen_logs]
        chosen   = [l['rewards/chosen'] for l in chosen_logs]
        rejected = [l['rewards/rejected'] for l in rejected_logs]
        axes[2].plot(steps, chosen,   color='#2ecc71', label='Chosen',   linewidth=1.5)
        axes[2].plot(steps, rejected, color='#e74c3c', label='Rejected', linewidth=1.5)
        axes[2].set_title('Reward Scores', fontweight='bold')
        axes[2].set_xlabel('Step')
        axes[2].set_ylabel('Reward')
        axes[2].legend()

    plt.suptitle('DPO Training Curves', fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(f'{DPO_OUTPUT_DIR}/training_curves.png', bbox_inches='tight', dpi=120)
    plt.show()
    print('Training curves saved.')

## 9. Save DPO Adapter

In [ ]:
DPO_ADAPTER_DIR = '/kaggle/working/dpo_adapter'
trainer.model.save_pretrained(DPO_ADAPTER_DIR)
tokenizer.save_pretrained(DPO_ADAPTER_DIR)

adapter_size = sum(
    os.path.getsize(os.path.join(DPO_ADAPTER_DIR, f))
    for f in os.listdir(DPO_ADAPTER_DIR)
) / 1e6

print(f'DPO adapter saved : {DPO_ADAPTER_DIR}')
print(f'Adapter size      : {adapter_size:.1f} MB')
print(f'Files             : {os.listdir(DPO_ADAPTER_DIR)}')

## 10. Quick Inference Comparison

Side-by-side comparison of SFT vs DPO on the same prompts.
This is a qualitative sanity check before the full evaluation in notebook 04.

In [ ]:
def generate(model, tokenizer, prompt, max_new_tokens=150):
    formatted = (
        f'<|begin_of_text|>'
        f'<|start_header_id|>user<|end_header_id|>\n\n{prompt}<|eot_id|>'
        f'<|start_header_id|>assistant<|end_header_id|>\n\n'
    )
    inputs = tokenizer(formatted, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = out[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

test_prompts = [
    'What are the most important things to consider when choosing a programming language for a new project?',
    'Explain gradient descent to someone who has never studied machine learning.',
    'Give me practical advice for managing stress during a busy workweek.',
]

print('=== QUALITATIVE COMPARISON: DPO vs SFT ===')
print('(Full win-rate evaluation with GPT-4o judge is in notebook 04)')
print()

for i, prompt in enumerate(test_prompts, 1):
    dpo_response = generate(trainer.model, tokenizer, prompt)
    print(f'{'='*65}')
    print(f'PROMPT {i}: {prompt}')
    print(f'{'='*65}')
    print(f'DPO Response:')
    print(f'  {dpo_response[:400]}')
    print()

## 11. Training Summary

In [ ]:
# Extract final reward metrics if available
final_train = train_logs[-1] if train_logs else {}
final_margin  = final_train.get('rewards/margins', 'N/A')
final_chosen  = final_train.get('rewards/chosen',  'N/A')
final_rejected= final_train.get('rewards/rejected','N/A')

print('=' * 65)
print(f'{"DPO TRAINING SUMMARY":^65}')
print('=' * 65)
print(f'  Base model      : {BASE_MODEL_ID}')
print(f'  Dataset         : UltraFeedback 10K preference pairs')
print(f'  Beta            : {BETA}')
print(f'  LoRA rank       : {LORA_RANK}')
print(f'  Final loss      : {train_result.training_loss:.4f}')
print(f'  Reward margin   : {final_margin}')
print(f'  Chosen reward   : {final_chosen}')
print(f'  Rejected reward : {final_rejected}')
print(f'  Adapter saved   : {DPO_ADAPTER_DIR}')
print()
print('  NEXT STEPS:')
print('  1. Download dpo_adapter/ and training_logs.json')
print('  2. Open notebook 04_evaluation.ipynb on your Mac')
print('  3. Run win-rate evaluation with GPT-4o as judge')
print('  4. Run best-of-N test-time scaling experiment')
print('  5. Plot ablation study results')
print('=' * 65)